# Track a watershed through a drought

**Terrain intermediate project · Python · USGS Water Data**

Follow daily streamflow at multiple USGS gages across Southern California before, during, and after recent drought years.

**You will build:** a multi-site hydrograph dashboard with drought periods shaded.

**Before you start**

- Runs in [Google Colab](https://colab.research.google.com) or Jupyter.
- Uses live **USGS NWIS** daily discharge (parameter `00060`) — no API key required.
- Read the [USGS Water explainer](../explainer.html?id=usgs-water).

## 1. Setup

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from io import StringIO

plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})

## 2. Choose gages and drought window

These sites cover coastal Southern California watersheds. Adjust `SITES` or dates to explore your own region.

In [ ]:
SITES = {
    "11098000": "Arroyo Seco nr Pasadena",
    "11087020": "San Gabriel River ab Whittier Narrows Dam",
    "11113000": "Sespe Creek nr Fillmore",
}

START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

# Approximate severe drought years in California (for shading)
DROUGHT_YEARS = [2015, 2016, 2021, 2022]

## 3. Download daily discharge from USGS

In [ ]:
def fetch_usgs_daily(site_ids, start, end, param="00060"):
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": ",".join(site_ids),
        "parameterCd": param,
        "startDT": start,
        "endDT": end,
        "siteStatus": "all",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()

    rows = []
    for series in payload.get("value", {}).get("timeSeries", []):
        site = series["sourceInfo"]["siteCode"][0]["value"]
        name = series["sourceInfo"]["siteName"]
        values = series["values"][0]["value"]
        for v in values:
            rows.append({
                "site_id": site,
                "site_name": name,
                "date": v["dateTime"][:10],
                "cfs": float(v["value"]) if v["value"] not in ("-999999", "") else None,
            })
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df

flow = fetch_usgs_daily(list(SITES.keys()), START_DATE, END_DATE)
flow = flow.dropna(subset=["cfs"])
print(f"Downloaded {len(flow):,} daily records across {flow['site_id'].nunique()} sites")
flow.head()

## 4. Summarize by year (drought signal)

In [ ]:
annual = (
    flow.assign(year=flow["date"].dt.year)
        .groupby(["site_id", "site_name", "year"], as_index=False)["cfs"]
        .mean()
        .rename(columns={"cfs": "mean_cfs"})
)
annual.pivot(index="year", columns="site_name", values="mean_cfs").round(1)

## 5. Multi-site hydrograph dashboard

In [ ]:
fig, axes = plt.subplots(len(SITES), 1, figsize=(10, 3.2 * len(SITES)), sharex=True)
if len(SITES) == 1:
    axes = [axes]

for ax, (site_id, label) in zip(axes, SITES.items()):
    sub = flow[flow["site_id"] == site_id].sort_values("date")
    ax.plot(sub["date"], sub["cfs"], color="#2F6B4F", linewidth=0.9, label="Daily mean discharge")
    for yr in DROUGHT_YEARS:
        ax.axvspan(pd.Timestamp(f"{yr}-01-01"), pd.Timestamp(f"{yr}-12-31"),
                   color="#D98E5A", alpha=0.12)
    ax.set_ylabel("cfs")
    ax.set_title(label, loc="left", fontsize=12, fontweight="bold")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[-1].set_xlabel("Date")
fig.suptitle("Southern California streamflow through drought years (USGS daily discharge)",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

## 6. Write your takeaway

1. Which gage shows the largest drop during shaded drought years?
2. Did flows recover equally at all three sites after 2022?
3. Why shouldn't you compare raw `cfs` across gages of different watershed sizes without normalization?

**Next:** See [NOAA climate data](../explainer.html?id=noaa-climate) for precipitation context, or browse more on [Terrain](../datasets.html).